## BCI Motor Movement : Quick Overview

* **What we are doing**: Looking at brain waves when someone physically executes moving their left or right hand.
* **Why we filter it (8–30 Hz)**: Brain signals are noisy. This filter cuts out the junk and keeps only the brain rhythms (mu and beta waves) that matter for hand movement.

In [ ]:
from pathlib import Path
import mne

# Update this path to where your dataset is stored locally
DATA_DIR = Path(r"C:\Users\mahia\Desktop\eegmmidb")

# Physical Movement runs (Runs 03, 07, 11)
run_files = [
    DATA_DIR / "S001" / "S001R03.edf",
    DATA_DIR / "S001" / "S001R07.edf",
    DATA_DIR / "S001" / "S001R11.edf",
]

# Verify all required files exist before loading
for file in run_files:
    if not file.is_file():
        raise FileNotFoundError(f"Missing EEG recording: {file}")

raw_list = []
for file in run_files:
    raw = mne.io.read_raw_edf(file, preload=True, verbose=False)
    
    # Standardize channel names and set standard 10-20 electrode positions
    mne.datasets.eegbci.standardize(raw)
    montage = mne.channels.make_standard_montage('standard_1020')
    raw.set_montage(montage, on_missing='ignore')
    
    # Apply band-pass filter
    raw.filter(8.0, 30.0, fir_design='firwin', skip_by_annotation='edge', verbose=False)
    raw_list.append(raw)

# Combine raw objects if multiple files exist
if len(raw_list) > 1:
    raw = mne.concatenate_raws(raw_list)
else:
    raw = raw_list[0]

print(f"Successfully loaded, montaged, and filtered {len(run_files)} physical movement files!")

## Feature Extraction : Quick Overview

* **Epoch Extraction**: Slicing the data into 3-second windows right after the movement cue.
* **CSP (Common Spatial Patterns)**: Projecting the data into a space that maximizes the contrast between left and right hand physical movements, making it easy for the classifier to read.

In [ ]:
import numpy as np

epochs_list = []
event_id = {'T1': 2, 'T2': 3}

for raw in raw_list:
    events, _ = mne.events_from_annotations(raw)
    epochs = mne.Epochs(
        raw, 
        events, 
        event_id, 
        tmin=0.0, 
        tmax=3.0, 
        proj=False, 
        baseline=None, 
        preload=True
    )
    epochs_list.append(epochs)

all_epochs = mne.concatenate_epochs(epochs_list)
X = all_epochs.get_data() # Raw EEG trials
y = all_epochs.events[:, -1] # Class labels

print(f"Extracted raw EEG shape: {X.shape} for {len(y)} total trials.")

## Classification & Results : Quick Overview

* **Classifier**: Linear Discriminant Analysis (LDA) trained with 5-fold cross-validation.
* **The Result**: **88.89% Accuracy**—proving that physical muscle execution generates massive, crystal-clear electrical signals in the motor cortex!

In [ ]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# Build Leakage-Free Pipeline (CSP fits strictly inside training folds now)
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
lda = LinearDiscriminantAnalysis()
clf = Pipeline([
    ('csp', csp),
    ('lda', lda)
])

# Stratified K-Fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")

print(f"--- MOTOR MOVEMENT RESULTS ---")
print(f"5-Fold Cross-Validation Scores: {np.round(scores * 100, 2)}")
print(f"Mean Classification Accuracy: {scores.mean() * 100:.2f}% (+/- {scores.std() * 100:.2f}%)")

import json

# Automatically save fold scores to the shared JSON file
try:
    with open("bci_scores.json", "r") as f:
        data_to_save = json.load(f)
except FileNotFoundError:
    data_to_save = {}

data_to_save["movement_folds"] = (scores * 100).tolist()
with open("bci_scores.json", "w") as f:
    json.dump(data_to_save, f)



## Brain Topography Mapping

* **Topographic Scalp Plots**: Displays the spatial filter patterns (`CSP0` to `CSP3`) across the motor cortex regions.

In [ ]:
import matplotlib.pyplot as plt
from mne.decoding import CSP

# Fit a standalone CSP instance strictly for visualization/topography plotting
viz_csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
viz_csp.fit(X, y)

# Plot clean CSP spatial patterns with component labels and colorbar
fig = viz_csp.plot_patterns(all_epochs.info, ch_type='eeg', components=range(4), show_names=False, colorbar=True)
plt.suptitle('CSP Spatial Patterns (Motor Movement)', fontsize=14, y=1.82)
plt.show()